# 46 · Analysis: AI exposure by major, and the skill-nearest low-exposure job with comparable pay

**Rules:** written before this notebook ran; decision D22 in `docs/decisions.md`. Reads: `dim_major_award` (link status, D20 field columns), `bridge_major_job` (D2 weights), `fct_degree_awards` (2024 first-major awards) with `dim_college` and `dim_major`, `dim_job` (D7 scores and bands), `fct_job_employment_wage_annual` (OEWS NY metro May 2025, group weights), `fct_job_skill` (O*NET importance ratings, 68 elements), `fct_job_wage_nyc` (NYC median wage), `fct_job_projection` 2022-32 (openings), `fct_grad_outcome` (`field_x_soc_major`, for the dispersed majors). Changes nothing.

**Questions**
1. How exposed to AI are the jobs each major leads to, and how are 2024 NYC awards (CUNY as a subset) spread across that exposure?
2. For each High-exposure degree-reachable job, which low-exposure degree-reachable jobs use similar knowledge and skills and pay about the same (the "escape routes")?

**Rules (§3i):** major exposure = `weight_openings`-weighted mean of `ai_observed_exposure` over the major's scored bridge jobs (coverage flagged under 70%), High share beside it, D7 bands on the mean; dispersed majors (D20) get the ACS destination-weighted group exposure instead, labeled. Similarity = profile correlation of the 68 O*NET importance ratings (D22); close ≥ 0.75, moderate 0.50–0.75; comparable pay = destination NYC median ≥ 90% of the origin's; destinations are degree-reachable, not All Other, exposure < 0.10, ≥ 100 projected openings a year; top three per origin.

**Limits stated with every number:** exposure is one company's observed use (Anthropic, Mar 2026), not job loss; O*NET ratings are national incumbent ratings; the NYC median wage is not an offer; similarity is not a transition rate (no NYC transition data); the bridge weights are the D2 openings convention.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from explore import q

pd.set_option("display.max_columns", 40, "display.max_colwidth", 52, "display.width", 250,
              "display.float_format", "{:,.2f}".format)
LEVEL = {3: "associate's", 5: "bachelor's"}
CLOSE, MODERATE, PAY_FLOOR, OPENINGS_FLOOR, EXPOSURE_CAP = 0.75, 0.50, 0.90, 100, 0.10


def md(s):
    display(Markdown(s))


def show(df, title=None):
    if title:
        md(f"**{title}**")
    display(df)


def band(x):
    return np.select([x.isna(), x == 0, x < 0.10, x < 0.30], ["Not rated", "Not seen yet", "Low", "Medium"], default="High")

## 1. Exposure per job and per job group
`dim_job` carries the D7 score and band. Group exposure (for the dispersed majors in §2c) is the NY-metro employment-weighted mean over the group's scored detailed jobs, the nb 45 §1 formula.

In [2]:
jobs = q("""
select j.soc_code, j.soc_title, j.major_code, j.entry_education, j.is_degree_entry, j.is_all_other,
       j.ai_observed_exposure, j.ai_band, j.ai_beta_gpt4,
       e.employment as emp_metro_2025, w.median_wage, w.entry_wage, p.annual_openings_avg, p.pct_change
from dim_job j
left join fct_job_employment_wage_annual e on e.soc_code = j.soc_code and e.area_key = 'ny_metro' and e.year = 2025 and e.occ_level = 'detailed'
left join fct_job_wage_nyc w on w.soc_code = j.soc_code
left join fct_job_projection p on p.soc_code = j.soc_code and p.edition = '2022-2032' and p.is_leaf
where j.soc_level = 'detailed'
""")
jobs["grp"] = jobs.soc_code.str[:2]
mg = q("select soc_code, soc_title from dim_job where soc_level = 'major'")
mg["grp"] = mg.soc_code.str[:2]
mg_title = mg.set_index("grp").soc_title.str.replace(" Occupations", "", regex=False)
scored = jobs[jobs.ai_observed_exposure.notna() & jobs.emp_metro_2025.notna()]
grp_exp = (scored.assign(wx=scored.emp_metro_2025 * scored.ai_observed_exposure).groupby("grp")
                 .agg(wx=("wx", "sum"), w=("emp_metro_2025", "sum")))
grp_exp["group exposure"] = grp_exp.wx / grp_exp.w
print(f"detailed jobs: {len(jobs)} | scored: {int(jobs.ai_observed_exposure.notna().sum())} | degree-reachable: {int(jobs.is_degree_entry.sum())}, "
      f"of which High band: {int((jobs.is_degree_entry & (jobs.ai_band == 'High')).sum())}, exposure < {EXPOSURE_CAP}: {int((jobs.is_degree_entry & (jobs.ai_observed_exposure < EXPOSURE_CAP)).sum())}")
show(jobs[jobs.is_degree_entry].groupby("ai_band").agg(jobs=("soc_code", "size"), openings=("annual_openings_avg", "sum"), median_wage=("median_wage", "median"))
       .reindex(["High", "Medium", "Low", "Not seen yet", "Not rated"]).round(0), "Degree-reachable jobs by D7 band (openings 2022-32 leaves, NYC median wage)")
show(grp_exp[["group exposure"]].join(mg_title.rename("group")).sort_values("group exposure", ascending=False).round(3).head(8), "Group exposure (nb 45 §1 formula), top 8")

detailed jobs: 867 | scored: 756 | degree-reachable: 221, of which High band: 33, exposure < 0.1: 115


**Degree-reachable jobs by D7 band (openings 2022-32 leaves, NYC median wage)**

,jobs,openings,median_wage
ai_band,,,
High,33,"38,322.00","117,582.00"
Medium,56,"52,738.00","104,156.00"
Low,60,"24,851.00","102,680.00"
Not seen yet,55,"10,260.00","89,628.00"
Not rated,17,"7,513.00","87,463.00"


**Group exposure (nb 45 §1 formula), top 8**

,group exposure,group
grp,,
15,0.35,Computer and Mathematical
43,0.33,Office and Administrative Support
13,0.32,Business and Financial Operations
41,0.28,Sales and Related
23,0.20,Legal
27,0.18,"Arts, Design, Entertainment, Sports, and Media"
25,0.18,Educational Instruction and Library
11,0.16,Management


## 2. Exposure per major
### 2a. Linked majors through the bridge (§3i)

In [3]:
dma = q("select major_award_key, cip_code, award_level, link_status, field_fod1p, field_lift, field_verdict from dim_major_award")
bridge = q("select major_award_key, cip_code, award_level, soc_code, weight_openings, weight_equal, is_all_other from bridge_major_job")
maj = q("select cip_code, cip_title_clean as major, cip_family_title_clean as family from dim_major").set_index("cip_code")
aw = q("""
select a.cip_code, a.award_level, c.is_cuny, sum(a.awards_total) as awards
from fct_degree_awards a join dim_college c using (unitid)
where a.year = 2024 and a.major_number = 1 and a.award_level in (3, 5)
group by 1, 2, 3
""")
awp = aw.pivot_table(index=["cip_code", "award_level"], columns="is_cuny", values="awards", aggfunc="sum").fillna(0)
awp.columns = ["awards_other" if not c else "awards_cuny" for c in awp.columns]
awp["awards"] = awp.sum(axis=1)
total_2024 = q("select sum(awards_total) s from fct_degree_awards where year = 2024 and major_number = 1 and award_level in (3, 5)").s.iloc[0]
M = dma.merge(awp.reset_index(), on=["cip_code", "award_level"], how="left").fillna({"awards": 0, "awards_cuny": 0, "awards_other": 0})
M["major"] = M.cip_code.map(maj.major)
M["family"] = M.cip_code.map(maj.family)
M["level"] = M.award_level.map(LEVEL)
print(f"dim_major_award rows: {len(M):,}; 2024 first-major awards on them {M.awards.sum():,.0f} vs fact total {total_2024:,.0f}")
show(M.groupby(["level", "link_status"]).agg(majors=("cip_code", "size"), awards=("awards", "sum"), cuny=("awards_cuny", "sum")).round(0), "Majors and 2024 awards by link status")

bridge["weight_openings"] = bridge.weight_openings.fillna(0.0)
bridge["weight_equal"] = bridge.weight_equal.fillna(0.0)
B = bridge.merge(jobs[["soc_code", "ai_observed_exposure", "ai_band", "median_wage"]], on="soc_code", how="left")
Bs = B[B.ai_observed_exposure.notna()]
ex = Bs.groupby("major_award_key").apply(lambda g: pd.Series({
    "coverage": g.weight_openings.sum(),
    "exposure": (g.weight_openings * g.ai_observed_exposure).sum() / g.weight_openings.sum() if g.weight_openings.sum() > 0 else np.nan,
    "high share": g.weight_openings[g.ai_band == "High"].sum() / g.weight_openings.sum() if g.weight_openings.sum() > 0 else np.nan,
    "exposure (equal weights)": (g.weight_equal * g.ai_observed_exposure).sum() / g.weight_equal.sum() if g.weight_equal.sum() > 0 else np.nan,
    "wage (weighted)": (g.weight_openings * g.median_wage).sum() / g.weight_openings[g.median_wage.notna()].sum() if g.weight_openings[g.median_wage.notna()].sum() > 0 else np.nan,
    "scored jobs": len(g),
}), include_groups=False)
M = M.merge(ex, left_on="major_award_key", right_index=True, how="left")
L = M[M.link_status == "linked"].copy()
L["low coverage"] = L.coverage < 0.70
L["band"] = band(L.exposure)
print(f"linked majors: {len(L)} | with an exposure: {int(L.exposure.notna().sum())} | coverage < 70% flagged: {int(L['low coverage'].sum())} | "
      f"median coverage {L.coverage.median():.0%}")
cols = ["cip_code", "major", "level", "awards", "awards_cuny", "exposure", "high share", "band", "exposure (equal weights)", "wage (weighted)", "coverage", "low coverage"]
for lv in ["bachelor's", "associate's"]:
    d = L[(L.level == lv) & L.exposure.notna()]
    show(d.nlargest(20, "awards")[cols].round(3).reset_index(drop=True), f"The 20 largest linked {lv} majors by 2024 awards, with the exposure of their jobs")
show(L[L.exposure.notna() & (L.awards >= 100)].nlargest(15, "exposure")[cols].round(3).reset_index(drop=True), "Most exposed linked majors (≥ 100 awards)")
show(L[L.exposure.notna() & (L.awards >= 100)].nsmallest(15, "exposure")[cols].round(3).reset_index(drop=True), "Least exposed linked majors (≥ 100 awards)")

dim_major_award rows: 4,650; 2024 first-major awards on them 73,225 vs fact total 73,225


**Majors and 2024 awards by link status**

majors    awards      cuny
level       link_status                                 
associate's linked               330  6,108.00  3,537.00
            not_in_crosswalk     377    513.00    434.00
            transfer_pathway    1618 11,869.00  7,431.00
bachelor's  dispersed             73 12,067.00  6,804.00
            linked              1457 37,380.00 15,504.00
            not_in_crosswalk     377  1,575.00    358.00
            transfer_pathway     418  3,713.00    771.00

linked majors: 1787 | with an exposure: 1741 | coverage < 70% flagged: 53 | median coverage 100%


**The 20 largest linked bachelor's majors by 2024 awards, with the exposure of their jobs**

...

,cip_code,major,level,awards,awards_cuny,exposure,high share,band,exposure (equal weights),wage (weighted),coverage,low coverage
0,51.3801,Registered Nursing/Registered Nurse,associate's,"1,539.00",730.00,0.06,0.00,Low,0.06,"127,613.00",1.00,False
1,51.0801,Medical/Clinical Assistant,associate's,494.00,48.00,0.05,0.00,Low,0.05,"50,337.00",1.00,False
2,11.0701,Computer Science,associate's,422.00,422.00,0.29,0.00,Medium,0.29,"87,178.00",1.00,False
3,43.0107,Criminal Justice/Police Science,associate's,382.00,247.00,0.12,0.00,Medium,0.12,"109,679.00",1.00,False
4,52.0302,Accounting Technology/Technician and Bookkeeping,associate's,306.00,292.00,0.31,1.00,High,0.31,"63,017.00",1.00,False
5,11.0101,"Computer and Information Sciences, General",associate's,237.00,209.00,0.34,1.00,High,0.34,"123,993.00",1.00,False
6,51.0602,Dental Hygiene/Hygienist,associate's,214.00,101.00,0.00,0.00,Not seen yet,0.00,"123,356.00",1.00,False
7,47.0607,Airframe Mechanics and Aircraft Maintenance Tech...,associate's,166.00,0.00,0.00,0.00,Not seen yet,0.00,"130,737.00",1.00,False
8,51.0910,Diagnostic Medical Sonography/Sonographer and Ul...,associate's,161.00,0.00,0.00,0.00,Not seen yet,0.00,"112,268.00",1.00,False
9,51.0907,Medical Radiologic Technology/Science - Radiatio...,associate's,136.00,136.00,0.02,0.00,Low,0.01,"91,880.90",0.99,False


**Most exposed linked majors (≥ 100 awards)**

,cip_code,major,level,awards,awards_cuny,exposure,high share,band,exposure (equal weights),wage (weighted),coverage,low coverage
0,52.1801,"Sales, Distribution, and Marketing Operations, G...",bachelor's,501.00,501.00,0.63,1.00,High,0.63,"83,175.00",1.00,False
1,52.1902,Fashion Merchandising,bachelor's,583.00,32.00,0.63,1.00,High,0.63,"83,175.00",1.00,False
2,30.7102,Business Analytics,bachelor's,103.00,0.00,0.47,0.60,High,0.39,"117,215.70",1.00,False
3,27.0305,Financial Mathematics,bachelor's,262.00,8.00,0.45,0.71,High,0.34,"137,360.79",1.00,False
4,52.1401,"Marketing/Marketing Management, General",bachelor's,337.00,21.00,0.44,0.73,High,0.25,"143,535.20",0.98,False
5,15.1201,Computer Engineering Technology/Technician,bachelor's,100.00,98.00,0.44,0.83,High,0.18,"128,291.54",1.00,False
6,27.9999,"Mathematics and Statistics, Other",bachelor's,170.00,0.00,0.42,0.90,High,0.36,"138,738.34",1.00,False
7,27.0301,"Applied Mathematics, General",bachelor's,140.00,27.00,0.36,0.74,High,0.28,"143,960.17",1.00,False
8,09.0101,Speech Communication and Rhetoric,bachelor's,250.00,103.00,0.36,0.60,High,0.25,"102,243.58",0.97,False
9,10.0304,"Animation, Interactive Technology, Video Graphic...",bachelor's,178.00,0.00,0.36,1.00,High,0.36,"104,119.00",1.00,False


**Least exposed linked majors (≥ 100 awards)**

,cip_code,major,level,awards,awards_cuny,exposure,high share,band,exposure (equal weights),wage (weighted),coverage,low coverage
0,31.0505,Exercise Science and Kinesiology,bachelor's,136.00,106.00,0.00,0.00,Not seen yet,0.00,"61,616.00",1.00,False
1,50.0408,Interior Design,bachelor's,177.00,0.00,0.00,0.00,Not seen yet,0.00,"79,630.00",1.00,False
2,50.0410,Illustration,bachelor's,535.00,0.00,0.00,0.00,Not seen yet,0.00,"89,317.00",0.45,True
3,51.0602,Dental Hygiene/Hygienist,associate's,214.00,101.00,0.00,0.00,Not seen yet,0.00,"123,356.00",1.00,False
4,51.0910,Diagnostic Medical Sonography/Sonographer and Ul...,associate's,161.00,0.00,0.00,0.00,Not seen yet,0.00,"112,268.00",1.00,False
5,51.3501,Massage Therapy/Therapeutic Massage,associate's,116.00,27.00,0.00,0.00,Not seen yet,0.00,"60,864.00",1.00,False
6,51.0909,Surgical Technology/Technologist,associate's,125.00,25.00,0.00,0.00,Not seen yet,0.00,"87,922.00",1.00,False
7,15.1303,Architectural Drafting and Architectural CAD/CADD,associate's,112.00,112.00,0.00,0.00,Not seen yet,0.00,"79,931.00",1.00,False
8,47.0607,Airframe Mechanics and Aircraft Maintenance Tech...,associate's,166.00,0.00,0.00,0.00,Not seen yet,0.00,"130,737.00",1.00,False
9,49.0309,Marine Science/Merchant Marine Officer,bachelor's,137.00,0.00,0.00,0.00,Not seen yet,0.00,"129,832.67",1.00,False


### 2b. Where the 2024 awards sit on the exposure scale

In [4]:
def spread(d, label):
    t = d.groupby("band").agg(majors=("cip_code", "size"), awards=("awards", "sum"), cuny=("awards_cuny", "sum")).reindex(["High", "Medium", "Low", "Not seen yet"]).fillna(0)
    t["awards %"] = t.awards / t.awards.sum() * 100
    t["cuny %"] = t.cuny / t.cuny.sum() * 100
    t.insert(0, "majors group", label)
    return t


Lx = L[L.exposure.notna()]
S2 = pd.concat([spread(Lx[Lx.level == "bachelor's"], "linked bachelor's"), spread(Lx[Lx.level == "associate's"], "linked associate's")])
show(S2.round(1), "2024 awards by the D7 band of the major's weighted exposure (linked majors with a score)")
for lv in ["bachelor's", "associate's"]:
    d = Lx[Lx.level == lv]
    for who, col in [("all NYC", "awards"), ("CUNY", "awards_cuny")]:
        if d[col].sum() > 0:
            print(f"{lv} · {who}: awards-weighted exposure {np.average(d.exposure, weights=d[col]):.3f}; awards-weighted High share {np.average(d['high share'], weights=d[col]):.1%}; "
                  f"equal-weights sensitivity {np.average(d['exposure (equal weights)'].fillna(d.exposure), weights=d[col]):.3f}")
dd = Lx[(Lx.level == "bachelor's") & Lx["wage (weighted)"].notna()]
print(f"bachelor's linked majors: Spearman between exposure and the weighted NYC median wage of their jobs = {dd.exposure.rank().corr(dd['wage (weighted)'].rank()):.2f} "
      f"(awards-weighted Pearson {np.cov(dd.exposure, dd['wage (weighted)'], aweights=dd.awards + 1)[0, 1] / np.sqrt(np.cov(dd.exposure, aweights=dd.awards + 1) * np.cov(dd['wage (weighted)'], aweights=dd.awards + 1)):.2f})")

**2024 awards by the D7 band of the major's weighted exposure (linked majors with a score)**

,majors group,majors,awards,cuny,awards %,cuny %
band,,,,,,
High,linked bachelor's,213,"6,799.00","2,639.00",18.20,17.00
Medium,linked bachelor's,332,"18,257.00","8,933.00",48.80,57.60
Low,linked bachelor's,686,"10,806.00","3,687.00",28.90,23.80
Not seen yet,linked bachelor's,192,"1,518.00",245.00,4.10,1.60
High,linked associate's,30,722.00,561.00,11.80,15.90
Medium,linked associate's,54,"1,219.00","1,062.00",20.00,30.00
Low,linked associate's,92,"2,792.00","1,420.00",45.70,40.10
Not seen yet,linked associate's,142,"1,375.00",494.00,22.50,14.00


bachelor's · all NYC: awards-weighted exposure 0.204; awards-weighted High share 20.9%; equal-weights sensitivity 0.194
bachelor's · CUNY: awards-weighted exposure 0.220; awards-weighted High share 21.1%; equal-weights sensitivity 0.211
associate's · all NYC: awards-weighted exposure 0.105; awards-weighted High share 11.7%; equal-weights sensitivity 0.102
associate's · CUNY: awards-weighted exposure 0.137; awards-weighted High share 15.7%; equal-weights sensitivity 0.133
bachelor's linked majors: Spearman between exposure and the weighted NYC median wage of their jobs = 0.20 (awards-weighted Pearson -0.01)


### 2c. Dispersed majors (D20): exposure by where their graduates actually work
Their bridge jobs are not where their graduates go, so the exposure is the ACS destination mix (field × SOC major group, bachelor's-only holders aged 22–34, unsuppressed rows renormalized) × the group exposure from §1.

In [5]:
fx = q("""
select group_1 as fod1p, group_2 as major_code, share_of_group, suppressed, n_unweighted
from fct_grad_outcome where cut = 'field_x_soc_major' and degree_level = 'bachelor' and age_band = '22-34'
""")
fx["grp"] = fx.major_code.str[:2]
fx = fx.merge(grp_exp[["group exposure"]], left_on="grp", right_index=True, how="left")
fxr = fx[~fx.suppressed & fx["group exposure"].notna()]
dex = fxr.groupby("fod1p").apply(lambda g: pd.Series({"acs coverage": g.share_of_group.sum(),
                                                       "exposure (ACS destinations)": (g.share_of_group * g["group exposure"]).sum() / g.share_of_group.sum()}), include_groups=False)
D = M[M.link_status == "dispersed"].merge(dex, left_on="field_fod1p", right_index=True, how="left")
D["band (ACS destinations)"] = band(D["exposure (ACS destinations)"])
D["bridge exposure (not used)"] = D.exposure
dcols = ["cip_code", "major", "field_fod1p", "field_lift", "awards", "awards_cuny", "exposure (ACS destinations)", "band (ACS destinations)", "acs coverage", "bridge exposure (not used)"]
print(f"dispersed majors: {len(D)} | with an ACS-destination exposure: {int(D['exposure (ACS destinations)'].notna().sum())} | awards {D.awards.sum():,.0f}")
show(D.nlargest(20, "awards")[dcols].round(3).reset_index(drop=True), "The 20 largest dispersed majors, exposure by where their field's graduates work")
dd2 = D[D["exposure (ACS destinations)"].notna()]
print(f"dispersed bachelor's majors, awards-weighted exposure by ACS destinations: {np.average(dd2['exposure (ACS destinations)'], weights=dd2.awards):.3f} "
      f"(their bridge jobs would have said {np.average(dd2.exposure.fillna(0), weights=dd2.awards):.3f}); all NY-metro workers, employment-weighted over the 22 groups: "
      f"{grp_exp.wx.sum() / grp_exp.w.sum():.3f} (nb 45: 0.139)")

dispersed majors: 73 | with an ACS-destination exposure: 73 | awards 12,067


**The 20 largest dispersed majors, exposure by where their field's graduates work**

,cip_code,major,field_fod1p,field_lift,awards,awards_cuny,exposure (ACS destinations),band (ACS destinations),acs coverage,bridge exposure (not used)
0,42.0101,"Psychology, General",5200,1.03,"4,129.00","2,927.00",0.21,Medium,0.88,0.16
1,52.0201,"Business Administration and Management, General",6203,1.36,"1,922.00","1,032.00",0.26,Medium,0.78,0.17
2,43.0103,Criminal Justice/Law Enforcement Administration,5301,1.40,"1,607.00","1,264.00",0.17,Medium,0.42,0.04
3,45.1001,"Political Science and Government, General",5506,1.19,"1,276.00",498.00,0.26,Medium,0.76,0.25
4,42.2799,"Research and Experimental Psychology, Other",5200,1.03,853.00,0.00,0.21,Medium,0.88,0.07
5,52.1301,Management Science,6203,1.36,821.00,0.00,0.26,Medium,0.78,0.19
6,42.2812,Forensic Psychology,5200,1.03,476.00,476.00,0.21,Medium,0.88,0.07
7,52.0101,"Business/Commerce, General",6200,1.19,415.00,318.00,0.28,Medium,0.78,0.14
8,52.0701,Entrepreneurship/Entrepreneurial Studies,6203,1.36,103.00,40.00,0.26,Medium,0.78,0.13
9,52.0213,Organizational Leadership,6203,1.36,80.00,0.00,0.26,Medium,0.78,0.20


dispersed bachelor's majors, awards-weighted exposure by ACS destinations: 0.224 (their bridge jobs would have said 0.145); all NY-metro workers, employment-weighted over the 22 groups: 0.146 (nb 45: 0.139)


## 3. Escape routes: the skill-nearest low-exposure job with comparable pay (D22)
### 3a. The similarity matrix
68 O*NET importance ratings per job (33 knowledge, 10 essential skills, 25 transferable skills). Similarity = Pearson correlation of the two jobs' 68-element profiles (row-centered cosine); squared Euclidean distance on the same ratings is the sensitivity.

In [6]:
sk = q("select soc_code, domain, element_id, element_name, value from fct_job_skill where scale = 'IM'")
X = sk.pivot(index="soc_code", columns="element_id", values="value")
assert X.shape[1] == 68 and X.notna().all().all(), X.shape
dom = sk.drop_duplicates("element_id").set_index("element_id").domain
Z = X.sub(X.mean(axis=1), axis=0)
Zn = Z.div(np.sqrt((Z ** 2).sum(axis=1)), axis=0)
R = Zn @ Zn.T                                                     # profile correlation, 790 × 790
sq = np.square(X.values[:, None, :] - X.values[None, :, :]).sum(axis=2)
E = pd.DataFrame(sq, index=X.index, columns=X.index)              # squared Euclidean distance
Rk = {}
for name, cols_ in [("knowledge", dom[dom == "knowledge"].index), ("skills", dom[dom != "knowledge"].index)]:
    Zd = X[cols_].sub(X[cols_].mean(axis=1), axis=0)
    Zd = Zd.div(np.sqrt((Zd ** 2).sum(axis=1)), axis=0)
    Rk[name] = Zd @ Zd.T
print(f"skill matrix {X.shape[0]} × {X.shape[1]}; R symmetric: {np.allclose(R.values, R.values.T)}; unit diagonal: {np.allclose(np.diag(R.values), 1)}; "
      f"off-diagonal r: mean {R.values[np.triu_indices(len(R), 1)].mean():.2f}, 90th pct {np.quantile(R.values[np.triu_indices(len(R), 1)], 0.9):.2f}, share ≥ 0.75: {(R.values[np.triu_indices(len(R), 1)] >= CLOSE).mean():.1%}")

skill matrix 790 × 68; R symmetric: True; unit diagonal: True; off-diagonal r: mean 0.52, 90th pct 0.77, share ≥ 0.75: 13.3%


### 3b. Origins, destinations, routes

In [7]:
J = jobs.set_index("soc_code")
origins = J[J.is_degree_entry & (J.ai_band == "High")]
dest = J[J.is_degree_entry & ~J.is_all_other & (J.ai_observed_exposure < EXPOSURE_CAP) & J.median_wage.notna() & (J.annual_openings_avg >= OPENINGS_FLOOR)]
origins_ok = origins[origins.index.isin(X.index)]
dest_ok = dest[dest.index.isin(X.index)]
print(f"origins (degree-reachable, High band): {len(origins)}, with a skill profile {len(origins_ok)} | destinations (degree-reachable, not All Other, exposure < {EXPOSURE_CAP}, "
      f"NYC median wage, ≥ {OPENINGS_FLOOR} openings): {len(dest)}, with a profile {len(dest_ok)}")


def routes(origin, pay_floor=PAY_FLOOR, measure="r", k=3):
    o = J.loc[origin]
    cand = dest_ok[(dest_ok.median_wage >= pay_floor * o.median_wage) & (dest_ok.index != origin)]
    if measure == "r":
        s = R.loc[origin, cand.index].sort_values(ascending=False)
    else:
        s = (-E.loc[origin, cand.index]).sort_values(ascending=False)
    return s.head(k)


rows = []
for o in origins_ok.sort_values("annual_openings_avg", ascending=False).index:
    oj = J.loc[o]
    if pd.isna(oj.median_wage):
        rows.append({"origin": o, "origin title": oj.soc_title, "exposure": oj.ai_observed_exposure, "origin wage": np.nan, "origin openings": oj.annual_openings_avg, "route": "no NYC wage"})
        continue
    s = routes(o)
    for rank, (d, r) in enumerate(s.items(), 1):
        dj = J.loc[d]
        rows.append({"origin": o, "origin title": oj.soc_title, "exposure": oj.ai_observed_exposure, "origin wage": oj.median_wage, "origin openings": oj.annual_openings_avg,
                     "route": rank, "destination": d, "destination title": dj.soc_title, "r": r, "r knowledge": Rk["knowledge"].loc[o, d], "r skills": Rk["skills"].loc[o, d],
                     "dest exposure": dj.ai_observed_exposure, "dest wage": dj.median_wage, "wage ratio": dj.median_wage / oj.median_wage, "dest openings": dj.annual_openings_avg,
                     "closeness": "close" if r >= CLOSE else ("moderate" if r >= MODERATE else "none")})
RT = pd.DataFrame(rows)
best = RT[RT.route == 1].copy()
show(RT[RT.route != "no NYC wage"][["origin", "origin title", "exposure", "origin wage", "origin openings", "route", "destination title", "r", "closeness", "dest exposure", "dest wage", "wage ratio", "dest openings", "r knowledge", "r skills"]]
       .round(3).reset_index(drop=True), "Routes: the three skill-nearest low-exposure degree-reachable jobs paying ≥ 90% of the origin's NYC median, per High-band origin (ranked by origin openings)")
show(best.closeness.value_counts().rename("origins").to_frame().T, f"Best route per origin: close (r ≥ {CLOSE}) / moderate / none")
show(best[["origin title", "exposure", "origin wage", "destination title", "r", "closeness", "dest wage", "dest exposure"]].round(3).reset_index(drop=True), "The best route per origin")

origins (degree-reachable, High band): 33, with a skill profile 31 | destinations (degree-reachable, not All Other, exposure < 0.1, NYC median wage, ≥ 100 openings): 51, with a profile 51


**Routes: the three skill-nearest low-exposure degree-reachable jobs paying ≥ 90% of the origin's NYC median, per High-band origin (ranked by origin openings)**

,origin,origin title,exposure,origin wage,origin openings,route,destination title,r,closeness,dest exposure,dest wage,wage ratio,dest openings,r knowledge,r skills
0,13-2011,Accountants and Auditors,0.35,"110,746.00","6,648.00",1,Financial Examiners,0.91,close,0.04,"138,527.00",1.25,"1,151.00",0.94,0.91
1,13-2011,Accountants and Auditors,0.35,"110,746.00","6,648.00",2,Insurance Underwriters,0.91,close,0.06,"100,781.00",0.91,251.00,0.83,0.97
2,13-2011,Accountants and Auditors,0.35,"110,746.00","6,648.00",3,Actuaries,0.90,close,0.05,"166,007.00",1.50,130.00,0.89,0.91
3,13-1161,Market Research Analysts and Marketing Specialists,0.65,"103,602.00","6,037.00",1,Insurance Underwriters,0.93,close,0.06,"100,781.00",0.97,251.00,0.85,0.97
4,13-1161,Market Research Analysts and Marketing Specialists,0.65,"103,602.00","6,037.00",2,Sales Managers,0.90,close,0.04,"231,387.00",2.23,"1,584.00",0.89,0.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19-4031,Chemical Technicians,0.32,"60,639.00",51.00,2,"Environmental Scientists and Specialists, Includ...",0.71,moderate,0.06,"91,814.00",1.51,157.00,0.72,0.70
85,19-4031,Chemical Technicians,0.32,"60,639.00",51.00,3,Electrical Engineers,0.71,moderate,0.06,"133,675.00",2.20,228.00,0.69,0.75
86,15-2099,"Mathematical Science Occupations, All Other",0.48,"96,713.00",1.36,1,"Environmental Scientists and Specialists, Includ...",0.80,close,0.06,"91,814.00",0.95,157.00,0.68,0.90
87,15-2099,"Mathematical Science Occupations, All Other",0.48,"96,713.00",1.36,2,Occupational Health and Safety Specialists,0.77,close,0.00,"98,706.00",1.02,260.00,0.71,0.87


**Best route per origin: close (r ≥ 0.75) / moderate / none**

closeness,close,moderate
origins,29,1


**The best route per origin**

,origin title,exposure,origin wage,destination title,r,closeness,dest wage,dest exposure
0,Accountants and Auditors,0.35,"110,746.00",Financial Examiners,0.91,close,"138,527.00",0.04
1,Market Research Analysts and Marketing Specialists,0.65,"103,602.00",Insurance Underwriters,0.93,close,"100,781.00",0.06
2,"Securities, Commodities, and Financial Services ...",0.44,"175,416.00",Sales Managers,0.93,close,"231,387.00",0.04
3,Financial Managers,0.39,"242,789.00",Sales Managers,0.92,close,"231,387.00",0.04
4,Human Resources Specialists,0.40,"97,527.00",Human Resources Managers,0.93,close,"198,037.00",0.00
5,Public Relations Specialists,0.45,"88,285.00",Fundraisers,0.93,close,"84,333.00",0.05
6,Marketing Managers,0.32,"205,522.00",Sales Managers,0.95,close,"231,387.00",0.04
7,Personal Financial Advisors,0.35,"176,937.00",Actuaries,0.89,close,"166,007.00",0.05
8,Graphic Designers,0.37,"84,433.00",Music Directors and Composers,0.81,close,"100,710.00",0.04
9,Data Scientists,0.46,"140,079.00",Actuaries,0.84,close,"166,007.00",0.05


### 3c. Sensitivities: the pay floor, and Euclidean distance instead of correlation

In [8]:
tri = R.values[np.triu_indices(len(R), 1)]
q90, q95, q99 = np.quantile(tri, [0.90, 0.95, 0.99])
print(f"all 790-job pairs: r ≥ 0.75 is the top {(tri >= CLOSE).mean():.1%}; percentiles 90th {q90:.3f}, 95th {q95:.3f}, 99th {q99:.3f}")


def routes_band(origin, pay_floor, pay_ceiling=None, k=1):
    o = J.loc[origin]
    cand = dest_ok[(dest_ok.median_wage >= pay_floor * o.median_wage) & (dest_ok.index != origin)]
    if pay_ceiling is not None:
        cand = cand[cand.median_wage <= pay_ceiling * o.median_wage]
    return R.loc[origin, cand.index].sort_values(ascending=False).head(k)


sens = []
for label, pf, pc in [("100% of origin median, no ceiling", 1.00, None), ("90%, no ceiling (rule)", PAY_FLOOR, None), ("75%, no ceiling", 0.75, None), ("90% to 125% (lateral only)", PAY_FLOOR, 1.25)]:
    vals = pd.Series({o: (lambda s: s.iloc[0] if len(s) else np.nan)(routes_band(o, pf, pc)) for o in origins_ok.index if pd.notna(J.loc[o].median_wage)})
    sens.append({"pay band": label, "origins with a candidate": int(vals.notna().sum()), "close (≥ 0.75)": int((vals >= CLOSE).sum()), f"≥ 95th pct ({q95:.2f})": int((vals >= q95).sum()),
                 f"≥ 99th pct ({q99:.2f})": int((vals >= q99).sum()), "moderate": int(((vals >= MODERATE) & (vals < CLOSE)).sum()), "none": int((vals < MODERATE).sum()), "median best r": vals.median()})
show(pd.DataFrame(sens).round(3), "Pay-band and cutoff sensitivity (correlation measure, best route per origin)")
agree = []
for o in origins_ok.index:
    if pd.isna(J.loc[o].median_wage):
        continue
    r1 = routes(o).index[:1]
    e1 = routes(o, measure="euclid").index[:1]
    e3 = routes(o, measure="euclid").index
    agree.append({"origin title": J.loc[o].soc_title, "r nearest": J.loc[r1[0]].soc_title if len(r1) else None, "euclid nearest": J.loc[e1[0]].soc_title if len(e1) else None,
                  "same": bool(len(r1) and len(e1) and r1[0] == e1[0]), "r nearest in euclid top 3": bool(len(r1) and r1[0] in set(e3))})
AG = pd.DataFrame(agree)
print(f"origins where the Euclidean nearest is the correlation nearest: {int(AG.same.sum())} of {len(AG)}; correlation nearest inside the Euclidean top three: {int(AG['r nearest in euclid top 3'].sum())} of {len(AG)}")
show(AG[~AG.same].reset_index(drop=True), "Where the two measures disagree on the nearest route")

all 790-job pairs: r ≥ 0.75 is the top 13.3%; percentiles 90th 0.775, 95th 0.821, 99th 0.889


**Pay-band and cutoff sensitivity (correlation measure, best route per origin)**

,pay band,origins with a candidate,close (≥ 0.75),≥ 95th pct (0.82),≥ 99th pct (0.89),moderate,none,median best r
0,"100% of origin median, no ceiling",30,24,15,9,5,1,0.82
1,"90%, no ceiling (rule)",30,29,17,10,1,0,0.84
2,"75%, no ceiling",30,29,17,10,1,0,0.84
3,90% to 125% (lateral only),30,28,14,8,2,0,0.82


origins where the Euclidean nearest is the correlation nearest: 23 of 30; correlation nearest inside the Euclidean top three: 28 of 30


**Where the two measures disagree on the nearest route**

,origin title,r nearest,euclid nearest,same,r nearest in euclid top 3
0,Human Resources Specialists,Human Resources Managers,Labor Relations Specialists,False,False
1,Accountants and Auditors,Financial Examiners,Insurance Underwriters,False,True
2,Social Science Research Assistants,"Adult Basic Education, Adult Secondary Education...",Insurance Underwriters,False,True
3,"Fine Artists, Including Painters, Sculptors, and...",Interior Designers,Architectural and Civil Drafters,False,True
4,Interpreters and Translators,"Special Education Teachers, Secondary School",Insurance Underwriters,False,True
5,Sales Engineers,Sales Managers,Architectural and Engineering Managers,False,True
6,Desktop Publishers,Producers and Directors,Fashion Designers,False,False


## 4. Checks

In [9]:
chk = []
chk.append(["skill matrix 790 × 68, no nulls; R symmetric, unit diagonal", f"{X.shape}, nulls {int(X.isna().sum().sum())}; {np.allclose(R.values, R.values.T)}, {np.allclose(np.diag(R.values), 1)}"])
o1, d1 = best.iloc[0].origin, best.iloc[0].destination
chk.append([f"one pair by np.corrcoef: {o1} vs {d1}", f"{np.corrcoef(X.loc[o1], X.loc[d1])[0, 1]:.4f} vs R {R.loc[o1, d1]:.4f}"])
cs = L[(L.cip_code == "11.0701") & (L.award_level == 5)]
sqlx = q("""select sum(b.weight_openings * j.ai_observed_exposure) / sum(b.weight_openings) x, sum(b.weight_openings) cov
            from bridge_major_job b join dim_job j using (soc_code) where b.cip_code = '11.0701' and b.award_level = 5 and j.ai_observed_exposure is not null""")
chk.append(["Computer Science bachelor's exposure recomputed in SQL", f"{cs.exposure.iloc[0]:.4f} / cov {cs.coverage.iloc[0]:.3f} vs SQL {sqlx['x'].iloc[0]:.4f} / {sqlx['cov'].iloc[0]:.3f}" if len(cs) else "11.0701 not linked"])
chk.append(["2024 awards on dim_major_award vs fact total", f"{M.awards.sum():,.0f} vs {total_2024:,.0f}"])
chk.append(["awards by status: linked + dispersed + other = total", f"{L.awards.sum():,.0f} + {D.awards.sum():,.0f} + {M[~M.link_status.isin(['linked', 'dispersed'])].awards.sum():,.0f} = {M.awards.sum():,.0f}"])
chk.append(["weights sum to 1 per linked major with openings (max |Σw − 1|)", f"{(bridge[bridge.major_award_key.isin(L.major_award_key)].groupby('major_award_key').weight_openings.sum() - 1).abs().max():.6f}"])
chk.append(["Euclidean nearest = correlation nearest", f"{int(AG.same.sum())} of {len(AG)} origins"])
chk.append(["origins without a NYC wage (no route computed)", ", ".join(RT[RT.route == 'no NYC wage']['origin title']) or "none"])
show(pd.DataFrame(chk, columns=["check", "result"]), "Checks")

**Checks**

,check,result
0,"skill matrix 790 × 68, no nulls; R symmetric, un...","(790, 68), nulls 0; True, True"
1,one pair by np.corrcoef: 13-2011 vs 13-2061,0.9125 vs R 0.9125
2,Computer Science bachelor's exposure recomputed ...,0.3203 / cov 1.000 vs SQL 0.3203 / 1.000
3,2024 awards on dim_major_award vs fact total,"73,225 vs 73,225"
4,awards by status: linked + dispersed + other = t...,"43,488 + 12,067 + 17,670 = 73,225"
5,weights sum to 1 per linked major with openings ...,1.000000
6,Euclidean nearest = correlation nearest,23 of 30 origins
7,origins without a NYC wage (no route computed),Statistical Assistants


## 5. Findings (from the tables above)

1. Where NYC's 2024 awards sit on the exposure scale (§2a–2b). Linked bachelor's majors (37,380 awards): awards-weighted observed exposure 0.204 (CUNY 0.220), with 18% of awards in majors whose jobs average High (6,799; CUNY 17%), 49% Medium, 29% Low, 4% Not seen yet; the awards-weighted High share of the jobs themselves is 21%. Linked associate's majors (6,108): 0.105 (CUNY 0.137), 68% of awards in Low or Not seen yet (nursing, allied health, the trades). Equal weights move the numbers by 0.01, so the D2 convention is not driving this. Most exposed majors with 100+ awards: Sales, Distribution and Marketing Operations 0.63 (CUNY 501 awards, every job High, through Market Research Analysts), Fashion Merchandising 0.63, Business Analytics 0.47, Financial Mathematics 0.45, Marketing 0.44, Computer Engineering Technology 0.44, Mathematics and Statistics Other 0.42, Applied Mathematics 0.36, Fine/Studio Arts 0.36 (Fine Artists; coverage 79%), Web/Multimedia 0.33. Computer Science bachelor's 0.32 (High band, 28% of its weight in High jobs); Accounting 0.28, Medium on the mean but 69% of its weight in High jobs (Accountants 0.35); Econometrics 0.28 / 53%; Finance 0.25 / 35%. Least exposed: Nursing 0.06 at both levels, Dental Hygiene, Sonography, Surgical Technology, Exercise Science, Interior Design and Architectural Drafting at 0, Civil Engineering 0.01, Architectural Technology 0.01. Exposure and the pay of a major's jobs are unrelated across bachelor's majors (Spearman 0.20, awards-weighted Pearson −0.01): the exposed majors are not the low-paid ones.
2. **Dispersed majors (§2c, D20).** By where their graduates actually work (ACS field × job family × nb 45 group exposure), the 73 dispersed bachelor's majors (12,067 awards) sit at 0.224 awards-weighted, above their bridge jobs' 0.145 and above the all-worker 0.146: Business Administration 0.26, Political Science 0.26, Psychology 0.21, Criminal Justice 0.17 (only 42% of its destinations readable, so that one is a caution). The Management, Business and Financial, and Office families they land in are mid-to-high exposure; the pages for these majors should say that rather than a bridge number.
3. **Escape routes (§3, D22).** 33 degree-reachable jobs are High band; they carry 38,322 projected openings a year (28% of degree-reachable openings) and a median NYC wage of $117.6 k, above every other band. 31 have an O*NET profile and 30 a NYC wage (Statistical Assistants does not); 51 degree-reachable jobs qualify as destinations (exposure < 0.10, not All Other, wage, ≥ 100 openings). 29 of 30 origins have a "close" route at r ≥ 0.75 under the −10% pay floor. The best routes read well where the knowledge base is shared: Accountants → Financial Examiners (r 0.91, $139 k, exposure 0.04, 1,151 openings a year), Insurance Underwriters, Actuaries; Market Research Analysts → Insurance Underwriters 0.93, Sales Managers 0.90; Financial Managers and Marketing Managers → Sales Managers (0.92, 0.95); HR Specialists and Training Managers → HR Managers (0.93, 0.92); PR Specialists → Fundraisers 0.93; HR Assistants → Compensation and Benefits Specialists 0.92; Data Scientists, Operations Research Analysts, Personal Financial Advisors and Database Architects → Actuaries (0.78–0.89, $166 k, but only 130 openings a year). Tech origins have a thin destination set: Computer Programmers (0.75), QA Analysts (0.80), Database Administrators (0.82), Computer Occupations All Other (0.84) and Network Administrators (0.65, the one "moderate") all land on Electrical Engineers, because few low-exposure degree-reachable jobs share a computing profile. Some routes look wrong on their face (Graphic Designers → Music Directors and Composers 0.81; Interpreters → Special Education Teachers 0.86; Information Security Analysts → Administrative Services Managers 0.80; Desktop Publishers → Producers and Directors 0.83): the 68-element profile measures the shape of knowledge and skill importance, not what the job does, and it is a loose measure, with a mean pair correlation of 0.52 over all 790 jobs and 13% of all pairs at or above 0.75. At the 95th percentile of all pairs (0.82) 17 of 30 origins keep a route, at the 99th (0.89) 10. Sensitivities: squared Euclidean distance picks the same nearest job for 23 of 30 origins (28 of 30 within its top three); a no-pay-cut floor leaves 24 close routes, a lateral band (90–125%) 28; r on knowledge only and on skills only is shown beside each route (the knowledge correlation is the one that separates Accountants → Financial Examiners 0.94 from Accountants → Underwriters 0.83).
4. **Checks.** Skill matrix 790 × 68 with no nulls, R symmetric with a unit diagonal; Accountants → Financial Examiners recomputed with `np.corrcoef` (0.9125); Computer Science bachelor's exposure recomputed in SQL (0.3203, coverage 1.000); 2024 awards 73,225 = the fact total, split 43,488 linked + 12,067 dispersed + 17,670 transfer / not in crosswalk; openings weights sum to 1 on every linked major.
5. **Limits.** Exposure is Anthropic's observed use of one assistant (Mar 2026) and not job loss; the per-major number is a weighted mean of a bimodal set of jobs (show the High share); O*NET ratings are national incumbent ratings; the NYC median is not an offer; similarity is not a transition rate (no NYC transition data); the routes use knowledge and skills only, not tasks, abilities or work activities.
6. **For the mart.** (a) The major page gets the exposure band, the High share and the three most exposed jobs it leads to; dispersed majors get the ACS-destination version, labeled. (b) Recommendation on D22: keep the measure but raise the page cutoff from the pre-registered 0.75 to 0.82 (the 95th percentile of all job pairs) and label 0.75–0.82 "possible"; show the wage ratio so a 2× move (Sales Managers) reads as a promotion, not a lateral; cap at three routes. That change is logged as a proposed amendment to D22, to confirm. (c) The v2 upgrade is the O*NET Related Occupations file (task-based, the measure NYS DOL uses), which would fix the face-invalid pairs; a bridge follow-up is the thin tech destination set.